<a href="https://colab.research.google.com/github/beverlygaruba/Data-Analysis-and-Visualization-Assessment/blob/main/Data_Analysis_and_Visualisation_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os, re, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")          # keep the output readable
pd.set_option("display.max_columns", 40)   # show all columns when printing a DataFrame
pd.set_option("display.width", 160)


PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = PALETTE


try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

In [4]:

LOCAL_FILE = "calls_for_service_2026.csv"
URL = "https://data.nola.gov/api/views/es9j-6y5d/rows.csv?accessType=DOWNLOAD"

if os.path.exists(LOCAL_FILE):
    raw = pd.read_csv(LOCAL_FILE, dtype=str)
    print(f"Loaded local snapshot: {LOCAL_FILE}")
else:
    raw = pd.read_csv(URL, dtype=str)
    print("Loaded live data from data.nola.gov")

print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

Loaded live data from data.nola.gov
Raw shape: 290,217 rows x 21 columns


,NOPD_Item,Type,TypeText,Priority,InitialType,InitialTypeText,InitialPriority,MapX,MapY,TimeCreate,TimeDispatch,TimeArrive,TimeClosed,Disposition,DispositionText,SelfInitiated,Beat,BLOCK_ADDRESS,Zip,PoliceDistrict,Location
0,A0834626,58,RETURN FOR ADDITIONAL INFO,1I,58,RETURN FOR ADDITIONAL INFO,1I,3692982,529575,2026-01-09T05:36:16.963000,NaN,NaN,2026-01-09T05:38:35.913000,DUP,DUPLICATE,N,4H02,024XX Sanctuary Dr,70114,4,POINT (-90.03188343 29.94979537)
1,A0894326,29U,UNCLASSIFIED DEATH,1D,NOPD,INCIDENT REQUESTED BY ANOTHER AGENCY,2A,3698396,551547,2026-01-09T17:41:26.880000,2026-01-09T20:18:11.570000,2026-01-09T20:23:19.567000,2026-01-10T00:23:41.913000,RTF,REPORT TO FOLLOW,N,7C01,063XX Chef Menteur Hwy,70126,7,POINT (-90.01398923 30.01003994)
2,A0935026,64G,ARMED ROBBERY WITH GUN,2C,64G,ARMED ROBBERY WITH GUN,2C,3681307,535916,2026-01-10T03:46:16.677000,2026-01-10T03:48:29.147000,2026-01-10T03:52:11.200000,2026-01-10T06:57:17.437000,RTF,REPORT TO FOLLOW,N,1F02,015XX Governor Nicholls St,70116,1,POINT (-90.06852392 29.96759174)
3,A0971726,67A,AUTO THEFT,0,67A,AUTO THEFT,0,3672282,538047,2026-01-10T14:19:34.873000,NaN,2026-01-10T15:04:23.887000,2026-01-10T16:42:23.317000,RTF,REPORT TO FOLLOW,N,1M02,038XX Iberville St,70119,1,POINT (-90.09695229 29.97371974)
4,A1026326,21,COMPLAINT OTHER,1A,21,COMPLAINT OTHER,1J,3696511,524009,2026-01-11T02:53:12.250000,2026-01-12T03:17:18.433000,2026-01-12T03:17:20.360000,2026-01-12T04:59:14.007000,DUP,DUPLICATE,N,4J02,033XX Wall Blvd,70114,4,POINT (-90.02094002 29.93437875)


In [5]:
#Data-quality audit
missing = (raw.isna().sum()
              .to_frame("missing_count")
              .assign(missing_pct=lambda d: (d.missing_count / len(raw) * 100).round(2))
              .sort_values("missing_count", ascending=False))
missing[missing.missing_count > 0]

,missing_count,missing_pct
TimeDispatch,169357,58.36
TimeArrive,44618,15.37
Zip,5014,1.73
Beat,2650,0.91
Disposition,3,0.00
DispositionText,3,0.00
BLOCK_ADDRESS,1,0.00


In [6]:
audit = {}

audit["Location = POINT (0 0)"] = (raw["Location"] == "POINT (0 0)").sum()
audit["MapX = 0 or placeholder 37369000"] = raw["MapX"].isin(["0", "37369000"]).sum()


audit["Priority without sub-priority letter"] = raw["Priority"].str.fullmatch(r"\d").sum()
audit["InitialPriority invalid (not digit+optional letter)"] = (~raw["InitialPriority"].str.fullmatch(r"\d[A-Z]?")).sum()


audit["PoliceDistrict = 0"] = (raw["PoliceDistrict"] == "0").sum()


audit["Disposition = TST (test incident)"] = (raw["Disposition"] == "TST").sum()


audit["DispositionText in mixed case"] = raw["DispositionText"].dropna().apply(lambda s: s != s.upper()).sum()


audit["Alarm call-type spelling variants"] = raw.loc[raw["TypeText"].str.contains("ALARM|ALRAM"), "TypeText"].nunique()


created = pd.to_datetime(raw["TimeCreate"], errors="coerce")
audit["Rows in incomplete month (Nov 2026)"] = (created >= "2026-11-01").sum()


audit["Exact duplicate rows"] = raw.duplicated().sum()
audit["Duplicate NOPD_Item IDs"] = raw["NOPD_Item"].duplicated().sum()

pd.Series(audit, name="rows affected").to_frame()

,rows affected
Location = POINT (0 0),14553
MapX = 0 or placeholder 37369000,14553
Priority without sub-priority letter,33894
InitialPriority invalid (not digit+optional letter),2
PoliceDistrict = 0,2773
Disposition = TST (test incident),11
DispositionText in mixed case,176727
Alarm call-type spelling variants,9
Rows in incomplete month (Nov 2026),922
Exact duplicate rows,0


In [7]:
raw.loc[raw["TypeText"].str.contains("ALARM|ALRAM"), "TypeText"].value_counts()

,count
TypeText,
"BURGLAR ALARM, SILENT",12719
HOLD UP ALARM,1544
"BURGLAR ALARM, SILENT, FAR NO RESPONSE",421
"BURGLAR ALARM, LOCAL",199
BURGLARY ALRAM - RESIDENTIAL,68
ALARM: COMMERCIAL - BURGLARY,50
BURGLARY ALARM - BUSINESS,45
PANIC/HOLDUP ALRAM - RESIDENTIAL,8
PANIC/HOLDUP ALARM - BUSINESS,5


In [8]:
t = raw[["TimeCreate", "TimeDispatch", "TimeArrive", "TimeClosed"]].apply(pd.to_datetime, errors="coerce")
resp_raw = (t.TimeArrive - t.TimeCreate).dt.total_seconds() / 60
dur_raw  = (t.TimeClosed - t.TimeCreate).dt.total_seconds() / 60

mask_public = raw["SelfInitiated"].eq("N")
print("Response time (create -> arrive), public calls, minutes:")
print(resp_raw[mask_public].describe(percentiles=[.5, .9, .95, .99, .999]).round(1))
print(f"\nNegative response times (arrival before call created): {(resp_raw[mask_public] < 0).sum()}")
print(f"Response times > 24 hours:                             {(resp_raw[mask_public] > 1440).sum()}")
print(f"Calls closed before they were created:                 {(dur_raw < 0).sum()}")

Response time (create -> arrive), public calls, minutes:
count    131960.0
mean         63.4
std         126.9
min         -44.3
50%          20.2
90%         165.3
95%         265.0
99%         569.6
99.9%      1314.2
max        5346.2
dtype: float64

Negative response times (arrival before call created): 5
Response times > 24 hours:                             96
Calls closed before they were created:                 182
